In [1]:
# Python Day 9 - RAG (Retrieval Augmented Generation)
# BTech AI/ML | Vacation Learning

In [2]:
!pip install langchain langchain-groq langchain-community chromadb sentence-transformers --quiet
print("All installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/2

In [6]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Connect to Groq
llm = ChatGroq(
    api_key="YOUR_API_KEY",
    model="llama-3.3-70b-versatile"
)
print("Ready for RAG!")


Ready for RAG!


In [7]:
# My study notes as text
notes = """
Operating Systems Notes - Aditya Halder

Process Scheduling:
FCFS (First Come First Served) is the simplest scheduling algorithm.
Processes are executed in the order they arrive. No preemption.
Advantage: Simple to implement.
Disadvantage: Long waiting time for short processes (convoy effect).

Round Robin scheduling assigns a fixed time quantum to each process.
Each process gets equal CPU time. Good for time sharing systems.
Time quantum is usually 10-100 milliseconds.

Deadlocks:
A deadlock occurs when two or more processes are waiting for each other
to release resources, creating a circular dependency.
Four conditions for deadlock: Mutual Exclusion, Hold and Wait,
No Preemption, Circular Wait.
Deadlock prevention removes one of these four conditions.

Memory Management:
Paging divides memory into fixed size frames.
Each process is divided into pages of the same size.
Page table maps logical addresses to physical addresses.
Virtual memory allows programs larger than physical memory to run.
"""

print("Notes loaded! Length:", len(notes), "characters")

Notes loaded! Length: 1008 characters


In [9]:
# Step 1 - Split notes into small chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
)
chunks = splitter.create_documents([notes])
print("Total chunks created:", len(chunks))
print("\nFirst chunk:")
print(chunks[0].page_content)

Total chunks created: 8

First chunk:
Operating Systems Notes - Aditya Halder


In [10]:
# Step 2 - Create embeddings and store in vector database
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Store chunks in Chroma vector database
vectorstore = Chroma.from_documents(chunks, embeddings)

print("Vector database created!")
print("Total vectors stored:", vectorstore._collection.count())

/tmp/ipykernel_7730/1513869189.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector database created!
Total vectors stored: 8


In [11]:
# Step 3 - Ask questions about your notes!
def ask_notes(question):
    # Find most relevant chunks
    relevant_chunks = vectorstore.similarity_search(question, k=3)
    context = "\n".join([chunk.page_content for chunk in relevant_chunks])

    # Send to AI with context
    prompt = f"""
    You are a helpful study assistant. Answer the question based ONLY on the notes provided.

    Notes:
    {context}

    Question: {question}

    Answer clearly and simply:
    """

    response = llm.invoke(prompt)
    return response.content

# Test it!
print("Q: What is deadlock?")
print(ask_notes("What is deadlock?"))
print("\n" + "="*50 + "\n")
print("Q: What are the advantages of Round Robin?")
print(ask_notes("What are the advantages of Round Robin?"))

Q: What is deadlock?
A deadlock occurs when two or more processes are waiting for each other to release resources, creating a circular dependency.


Q: What are the advantages of Round Robin?
The advantage of Round Robin scheduling is that it is simple to implement.


In [12]:
print(ask_notes("What are the four conditions for deadlock?"))
print(ask_notes("Explain paging in memory management"))
print(ask_notes("What is the disadvantage of FCFS?"))

The four conditions for deadlock are: 
1. Mutual Exclusion
2. Hold and Wait
3. No Preemption
4. Circular Wait.
Paging in memory management is a technique that divides memory into fixed-size frames, and each process is also divided into pages of the same size. This allows for a page table to map logical addresses to physical addresses, enabling efficient memory use.
The disadvantage of FCFS is the long waiting time for short processes, also known as the convoy effect.


In [14]:
# Interactive study assistant!
print("🎓 Your Personal Study Assistant is ready!")
print("Type 'quit' to exit\n")

while True:
    question = input("Ask anything about your notes: ")
    if question.lower() == "quit":
        break
    answer = ask_notes(question)
    print("\n📚 Answer:", answer)
    print("\n" + "-"*50 + "\n")

🎓 Your Personal Study Assistant is ready!
Type 'quit' to exit

Ask anything about your notes: what is fcfs and what are the advantages and disadvantages of it

📚 Answer: FCFS (First Come First Served) is a scheduling algorithm where processes are executed in the order they arrive, without preemption.

The advantages of FCFS are:
1. Simple to implement

The disadvantage of FCFS is:
1. Long waiting time for short processes (convoy effect)

--------------------------------------------------

Ask anything about your notes: quit
